# Notebook 4 — Our own study data (run after the user study)

**Goal** (README section 7.5):

1. **Cross-dataset test:** test the SWELL-trained models on our study participants, with and without personal calibration.
2. **Retrain** the condition model on SWELL + our study participants (LOSO, people kept separate).
3. **Personal models** for the two developers (time-blocked split: train on earlier days, test on the last day).
4. **Compare** generic vs personalised.

Until the study data exists this notebook just prints a message and skips every step, so it still runs top to bottom.

### Expected input: `ml/data/own/*.csv`
One row per minute, exported from the app (no titles, no URLs, no typed text):

| Column | Meaning |
|---|---|
| `participant` | `P1`…`P5` (study participants) or `D1`, `D2` (developers) — IDs, never names |
| `session` | session / day identifier, e.g. `2026-12-01` |
| `timestamp` | unix seconds, start of the minute |
| `keystrokes`, `error_keys`, `shortcut_keys`, `clicks`, `scrolls`, `mouse_distance`, `app_switches` | raw counts (same as `shared/features.py`) |
| `condition` | `neutral` / `interruptions` / `time_pressure` during study blocks, empty otherwise |
| `load_rating` | 1–5 self-rating that applies to this minute (block rating or check-in), empty otherwise |

Developers' data (`D*`) is **never** used as evidence that the model generalises (README 17.1).

In [1]:
# --- Find the project folder so the notebook works from any working directory ---
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until we find the folder that contains shared/features.py."""
    for folder in [start, *start.parents]:
        if (folder / "shared" / "features.py").exists():
            return folder
    raise FileNotFoundError("Could not find the Commit project folder (shared/features.py).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))  # lets us "import shared.features"

RAW_DIR = REPO_ROOT / "ml" / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "ml" / "data" / "processed"
REPORTS_DIR = REPO_ROOT / "ml" / "reports"
MODELS_DIR = REPO_ROOT / "shared" / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Project folder:", REPO_ROOT)

Project folder: /Users/matrix/Desktop/Commit


In [2]:
import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut

from shared import features as F

# Hide repeated library warnings (e.g. a test day with only one class). They do not change results.
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)

In [3]:
# --- Settings ---

# Folder with our own exported study data. Can be overridden with an environment variable
# (used to test this notebook on a synthetic file).
OWN_DATA_DIR = Path(os.environ.get("COMMIT_OWN_DATA_DIR", REPO_ROOT / "ml" / "data" / "own"))

STUDY_PREFIX = "P"        # study participants: independent test set
DEVELOPER_PREFIX = "D"    # developers: personal-model case study only

# Exported model specification (feature names, calibration length, model details).
FEATURE_SPEC = json.loads((MODELS_DIR / "features.json").read_text())
FEATURE_NAMES = FEATURE_SPEC["feature_names"]
CALIBRATION_ACTIVE_MINUTES = FEATURE_SPEC["normalisation"]["calibration_active_minutes"]
WINDOWS = tuple(FEATURE_SPEC["rolling_windows_minutes"])

own_files = sorted(OWN_DATA_DIR.glob("*.csv")) if OWN_DATA_DIR.exists() else []
HAVE_OWN_DATA = len(own_files) > 0
if HAVE_OWN_DATA:
    print("Found own data files:", [f.name for f in own_files])
else:
    print(f"No own study data found in {OWN_DATA_DIR}.")
    print("This is expected until the user study has been run. All steps below are skipped.")

No own study data found in /Users/matrix/Desktop/Commit/ml/data/own.
This is expected until the user study has been run. All steps below are skipped.


## 1. Load our data and build features (same code as SWELL and the app)

In [4]:
def load_own_data(files: list) -> pd.DataFrame:
    """
    Read and combine the exported per-minute CSV files.

    In:  list of CSV paths in the format described above.
    Out: one table, with `timestamp` as a datetime and a `recording` column (participant + session).
    """
    table = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
    table["timestamp"] = pd.to_datetime(table["timestamp"], unit="s")
    table["participant"] = table["participant"].astype(str)
    table["recording"] = table["participant"] + "_" + table["session"].astype(str)
    return table


def build_and_flag(table: pd.DataFrame) -> pd.DataFrame:
    """
    Build all features with shared/features.py and mark calibration / evaluation rows.

    In:  raw own data.
    Out: feature table with is_calibration and in_evaluation columns (like Notebook 2).
    """
    built = F.build_features(table, time_column="timestamp", group_column="recording", windows=WINDOWS)
    built["is_calibration"] = F.select_calibration_minutes(
        built, "timestamp", CALIBRATION_ACTIVE_MINUTES, person_column="participant"
    )
    built["in_evaluation"] = (built[F.IDLE_COLUMN] == 0) & ~built["is_calibration"]
    return built


if HAVE_OWN_DATA:
    own_raw = load_own_data(own_files)
    own_features = build_and_flag(own_raw)
    print("Rows:", len(own_features), " participants:", sorted(own_features["participant"].unique()))
    print(own_features.groupby("participant")[["in_evaluation"]].sum().rename(columns={"in_evaluation": "modelling rows"}).T)

## 2. Two ways to normalise

* **With personal calibration:** z-scores against each person's own first 15 active minutes (what the app does).
* **Without personal calibration:** z-scores against one **generic** baseline — the pooled calibration minutes of all 25 SWELL participants (an "average person").

In [5]:
def generic_swell_baseline() -> dict:
    """
    Baseline of an "average person": mean/std of the pooled SWELL calibration minutes (raw features).

    Out: baseline dict in the format of shared/features.compute_baseline.
    """
    swell_raw = pd.read_csv(PROCESSED_DIR / "swell_features_raw.csv")
    return F.compute_baseline(swell_raw[swell_raw["is_calibration"]], FEATURE_NAMES)


if HAVE_OWN_DATA:
    own_personal, _ = F.normalise_per_person(own_features, "participant", own_features["is_calibration"], FEATURE_NAMES)
    own_generic = F.apply_baseline(own_features, generic_swell_baseline(), FEATURE_NAMES)

## 3. Cross-dataset test: SWELL-trained models on our study participants

In [6]:
def score(true: pd.Series, predicted) -> dict:
    """Macro-F1, balanced accuracy and accuracy for one set of predictions."""
    return {
        "macro_f1": f1_score(true, predicted, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(true, predicted),
        "accuracy": accuracy_score(true, predicted),
        "rows": len(true),
    }


def rating_events(rows: pd.DataFrame) -> pd.Series:
    """
    One value per rating event (one block rating or one check-in).

    In:  minutes of ONE person that have a load_rating.
    Out: the ratings, counting each run of consecutive minutes with the same rating
         in the same session only once.
    Why: a rating covers many minutes. A median over minutes would be dominated by
         the longest blocks and often leave nothing "above the median".
    """
    ordered = rows.sort_values("timestamp")
    rating_changed = ordered["load_rating"] != ordered["load_rating"].shift()
    session_changed = ordered["session"] != ordered["session"].shift()
    return ordered.loc[rating_changed | session_changed, "load_rating"]


def load_label_from_rating(table: pd.DataFrame) -> pd.Series:
    """
    low/high load from the 1–5 self-rating, split at each person's own median rating
    (one value per rating event, like SWELL's one value per block).

    In:  rows that have a load_rating.
    Out: Series of "low"/"high".
    """
    labels = pd.Series("low", index=table.index)
    for _, rows in table.groupby("participant"):
        threshold = rating_events(rows).median()
        labels.loc[rows.index[rows["load_rating"] > threshold]] = "high"
    return labels


def cross_dataset_test(normalised: pd.DataFrame, setup_name: str) -> list:
    """
    Score the exported condition and load models on study participants' labelled rows.

    In:  normalised own feature table, a name for this setup.
    Out: list of result dicts (one per participant and target, plus pooled).
    """
    condition_model = joblib.load(MODELS_DIR / "condition_model.pkl")
    load_model = joblib.load(MODELS_DIR / "load_model.pkl")
    study = normalised[normalised["participant"].str.startswith(STUDY_PREFIX) & normalised["in_evaluation"]]
    results = []

    # Condition: rows recorded during labelled study blocks.
    with_condition = study[study["condition"].notna()]
    if len(with_condition):
        predicted = pd.Series(condition_model.predict(F.order_for_model(with_condition, FEATURE_NAMES)), index=with_condition.index)
        for person, rows in with_condition.groupby("participant"):
            results.append({"setup": setup_name, "target": "condition", "participant": person, **score(rows["condition"], predicted.loc[rows.index])})
        results.append({"setup": setup_name, "target": "condition", "participant": "all", **score(with_condition["condition"], predicted)})

    # Load: rows with a self-rating.
    with_rating = study[study["load_rating"].notna()]
    if len(with_rating):
        true_load = load_label_from_rating(with_rating)
        predicted = load_model.predict(F.order_for_model(with_rating, FEATURE_NAMES))
        results.append({"setup": setup_name, "target": "load", "participant": "all", **score(true_load, predicted)})
    return results


if HAVE_OWN_DATA:
    cross_results = pd.DataFrame(
        cross_dataset_test(own_personal, "with personal calibration")
        + cross_dataset_test(own_generic, "without personal calibration (generic baseline)")
    )
    cross_results.to_csv(REPORTS_DIR / "own_cross_dataset.csv", index=False)
    display(cross_results.round(3))

## 4. Retrain the condition model on SWELL + our study participants (LOSO)

Uses the same algorithm and hyperparameters as the exported model (from `features.json`), so this step
measures the effect of **more, more realistic data**, not of extra tuning.

In [7]:
def retrain_with_own_data(own_normalised: pd.DataFrame) -> pd.DataFrame:
    """
    LOSO over SWELL participants + our study participants together.

    In:  own feature table with personal calibration.
    Out: per-test-person scores with a "dataset" column (SWELL / own study).
    """
    swell = pd.read_csv(PROCESSED_DIR / "swell_features_calibration.csv")
    swell = swell[swell["in_evaluation"]].assign(dataset="SWELL")
    study = own_normalised[
        own_normalised["participant"].str.startswith(STUDY_PREFIX)
        & own_normalised["in_evaluation"]
        & own_normalised["condition"].notna()
    ].assign(dataset="own study")
    combined = pd.concat([swell, study], ignore_index=True)

    # Untrained copy of the exported pipeline = same algorithm and hyperparameters.
    template = clone(joblib.load(MODELS_DIR / "condition_model.pkl"))
    X, y, groups = combined[FEATURE_NAMES], combined["condition"], combined["participant"]

    rows = []
    for train_index, test_index in LeaveOneGroupOut().split(X, y, groups):
        model = clone(template).fit(X.iloc[train_index], y.iloc[train_index])
        test = combined.iloc[test_index]
        rows.append({"participant": test["participant"].iloc[0], "dataset": test["dataset"].iloc[0],
                     **score(test["condition"], model.predict(X.iloc[test_index]))})
    return pd.DataFrame(rows)


if HAVE_OWN_DATA:
    retrain_results = retrain_with_own_data(own_personal)
    retrain_results.to_csv(REPORTS_DIR / "own_retrained_loso.csv", index=False)
    display(retrain_results.groupby("dataset")[["macro_f1", "balanced_accuracy", "accuracy"]].agg(["mean", "std"]).round(3))

## 5. Personal models for the developers (time-blocked split)

For each developer: train on all sessions except the **last** one, test on the last one (never on minutes
from the same day). Labels come from the 1–5 check-in ratings (high = above the developer's median in the
training days). We compare the generic SWELL load model with a personal model of the same algorithm.

In [8]:
def personal_vs_generic(own_normalised: pd.DataFrame) -> pd.DataFrame:
    """
    Train a personal load model per developer on earlier sessions and compare it with the
    generic SWELL load model on the last session.

    In:  own feature table with personal calibration.
    Out: one row per developer and model type with its scores on the last session.
    """
    generic_model = joblib.load(MODELS_DIR / "load_model.pkl")
    developers = own_normalised[
        own_normalised["participant"].str.startswith(DEVELOPER_PREFIX)
        & own_normalised["in_evaluation"]
        & own_normalised["load_rating"].notna()
    ]
    rows = []
    for person, data in developers.groupby("participant"):
        # Order sessions by their first minute; the last session is the test set.
        session_order = data.groupby("session")["timestamp"].min().sort_values().index
        if len(session_order) < 2:
            print(f"{person}: needs at least 2 sessions for a time-blocked split; skipped.")
            continue
        train = data[data["session"].isin(session_order[:-1])]
        test = data[data["session"] == session_order[-1]]

        # Threshold from TRAINING days only (the future is unknown when training).
        threshold = rating_events(train).median()
        train_label = np.where(train["load_rating"] > threshold, "high", "low")
        test_label = np.where(test["load_rating"] > threshold, "high", "low")
        if len(set(train_label)) < 2:
            print(f"{person}: all training ratings fall on one side of the median; skipped.")
            continue

        personal_model = clone(generic_model).fit(train[FEATURE_NAMES], train_label)
        for model_type, model in [("generic (SWELL)", generic_model), ("personal", personal_model)]:
            rows.append({"participant": person, "model": model_type,
                         "train_sessions": len(session_order) - 1,
                         **score(pd.Series(test_label), model.predict(test[FEATURE_NAMES]))})
    return pd.DataFrame(rows)


if HAVE_OWN_DATA:
    personal_results = personal_vs_generic(own_personal)
    personal_results.to_csv(REPORTS_DIR / "own_personal_vs_generic.csv", index=False)
    display(personal_results.round(3))

## Summary

Fill in after the study: does the SWELL model transfer to our participants, how much does personal calibration
help, does adding our data help on new people, and do personal models beat the generic one for the developers?